# INF201 – Deliverable 2: Campus Coffee Shop Sales Pipeline

**Group members:** Mathias Gundersen, Jakub Mermer  
**GitHub repository:** https://github.com/saltonese/Del_2-G6_5.git

## Introduction and approach

This notebook implements an ETL pipeline for the cash-register logs of the Campus Coffee chain:

1. **Extract** – recursively find all `.log` files, read them with the correct encoding (UTF-8 or ISO-8859-1) and pick out the valid sales lines with a regular expression.
2. **Transform** – build a DataFrame, remove price glitches with NumPy, and correct NOK prices that were registered on the EUR scale (× 10).
3. **Load / report** – hourly revenue per item, a status pivot table and refund ratios, written to `output/`.

Tasks 1–4 are small, individually tested building blocks. Task 5 connects them together into one .

### Interpretation choices

The assignment leaves a few things open. These are the choices made here:


### How to run
1. `python generate_data_for_d2.py` (creates `data/`; `BASE_DIR` was changed to `Path("Data")` and `random.seed(42)` added so all group members get identical data).
2. Run this notebook from the same folder. Output files are written to `output/`.

In [12]:
import json
import re
import tempfile #used to make temporary files, usefull when testing along the way
from pathlib import Path #used to easyely path between files and directories

import matplotlib.pyplot as plt #used to plot graphs
import numpy as np 
import pandas as pd

print(Path.cwd())
DATA_DIR = Path.cwd().parent / "Data"
OUTPUT_DIR = Path("output")

EPSILON = 1e-6          # keeps the refund ratio finite when sales_count == 0
NOK_CORRECTION = 10.0   # NOK prices were registered on the EUR scale
OUTLIER_N_STD = 3.0     # |x - mean| > 3 * std  ->  hardware glitch

print(f"pandas {pd.__version__} | numpy {np.__version__}")
assert DATA_DIR.exists(), "Run `python generate_data_for_d2.py` first to create the Data/ folder."

c:\Users\mathi\StudieNMBU\PROG\Del_2-G6_5\Notebooks
pandas 2.3.3 | numpy 2.3.4


## Task 1 – File ingestion and regular expressions

### 1.1 Robust file reader
The file is read as raw **bytes** and decoded explicitly. Reading bytes (instead of `read_text`) also avoids platform-dependent newline translation.

* First try `utf-8` (strict). Valid UTF-8 text is easy to recognise: Latin-1 text containing `é` or `ø` is almost never valid UTF-8, so a wrong guess raises `UnicodeDecodeError`.
* If that fails, decode as `iso-8859-1`. This always succeeds because every byte value maps to a character – that is why it has to be the *second* attempt.

Note: pure-ASCII files decode identically in both encodings, so they are reported as `utf-8`. The test below uses `é` and `ø` to show that the fallback really works.

In [13]:
def read_log_file(path: str | Path) -> tuple[str, str]:
    """Read a text file whose encoding is either UTF-8 or ISO-8859-1.

    Returns
    -------
    (file_text_content, encoding_used)
    """
    path = Path(path)
    raw = path.read_bytes()
    try:
        return raw.decode("utf-8"), "utf-8"
    except UnicodeDecodeError:
        return raw.decode("iso-8859-1"), "iso-8859-1"

In [14]:
# Test: the same text saved with two different encodings
with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    (tmp / "utf8.log").write_bytes("Café ølbar\n".encode("utf-8"))
    (tmp / "latin1.log").write_bytes("Café ølbar\n".encode("iso-8859-1"))

    text_utf8, enc_utf8 = read_log_file(tmp / "utf8.log")
    text_latin1, enc_latin1 = read_log_file(tmp / "latin1.log")

assert (enc_utf8, enc_latin1) == ("utf-8", "iso-8859-1")
assert text_utf8 == text_latin1 == "Café ølbar\n"
print("read_log_file OK:", enc_utf8, "/", enc_latin1)

read_log_file OK: utf-8 / iso-8859-1


### 1.2 Log parser with regular expressions
One compiled pattern with **named groups** matches a complete valid line. Details:

* `^ … $` anchor the pattern to the whole line, so a noise line such as `[DEBUG] BARCODE SCANNER REBOOTED` (which also starts with `[`) cannot match by accident.
* `item=(?P<item>.+?)\s+price=` is lazy, so the item name can in principle contain spaces ("Iced Latte").
* `status` is restricted to the three allowed values, so a corrupted status makes the whole line invalid instead of slipping through.
* Noise lines simply do not match and are skipped; the price is converted to `float`.

In [15]:
LOG_PATTERN = re.compile(
    r"""
    ^\[(?P<timestamp>\d{4}-\d{2}-\d{2}\ \d{2}:\d{2}:\d{2})\]\s+   # [2026-10-15 08:30:00]
    \[(?P<register_id>[A-Za-z0-9_]+)\]\s+                       # [POS_01]
    item=(?P<item>.+?)\s+                                       # item=Latte
    price=(?P<price>\d+(?:\.\d+)?)\s+                           # price=4.50
    (?P<currency>[A-Z]{3})\s+                                   # EUR / NOK / USD
    \((?P<status>SUCCESS|REFUND|CANCELLED)\)$                   # (SUCCESS)
    """,
    re.VERBOSE,
)

# Only used for the validation at the end (the parser itself ignores everything that does not match)
NOISE_PATTERN = re.compile(r"^(---.*---|===.*===|\[DEBUG\].*)$")


def parse_log_text(text: str) -> list[dict]:
    """Extract all valid sales lines from log text, ignoring system noise.

    Each record has the keys: timestamp (str), register_id (str), item (str),
    price (float), currency (str), status (str).
    """
    records = []
    for line in text.splitlines():
        match = LOG_PATTERN.match(line.strip())
        if match is None:
            continue  # noise or malformed line
        record = match.groupdict()
        record["price"] = float(record["price"])
        records.append(record)
    return records

In [16]:
sample = """--- PRINTER PAPER LOW ---
[2026-10-15 08:30:00] [POS_01] item=Latte price=4.50 EUR (SUCCESS)
[DEBUG] BARCODE SCANNER REBOOTED
=== NETWORK SYNC OK ===
[2026-10-15 08:31:12] [POS_02] item=Muffin price=999.00 NOK (REFUND)
[2026-10-15 08:32:00] [POS_02] item=Muffin price=abc NOK (REFUND)
[2026-10-15 08:33:00] [POS_02] item=Muffin price=3.50 NOK (UNKNOWN)
"""

parsed = parse_log_text(sample)
assert len(parsed) == 2
assert parsed[0] == {
    "timestamp": "2026-10-15 08:30:00",
    "register_id": "POS_01",
    "item": "Latte",
    "price": 4.5,
    "currency": "EUR",
    "status": "SUCCESS",
}
assert parsed[1]["price"] == 999.0 and parsed[1]["status"] == "REFUND"
print("parse_log_text OK ->", len(parsed), "of 7 lines are valid")

parse_log_text OK -> 2 of 7 lines are valid


## Task 2 – Recursive utility functions

### 2.1 Recursive search for log files
Explicit recursion: for every entry in a directory, a sub-directory triggers a recursive call and a file with the right extension is collected. No `glob`, `rglob` or `os.walk`. The base case is implicit – a directory without sub-directories simply makes no further recursive calls.

`iterdir()` returns entries in arbitrary order, so the workflow sorts the result to make the pipeline deterministic.

In [17]:
def find_log_files(directory: str | Path, extension: str = ".log") -> list[Path]:
    """Recursively collect all files with the given extension below `directory`."""
    directory = Path(directory)
    found: list[Path] = []
    for item in directory.iterdir():
        if item.is_dir():
            found.extend(find_log_files(item, extension))   # recursive case
        elif item.is_file() and item.suffix == extension:
            found.append(item)
    return found

In [18]:
with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    for rel in ["2026/a/POS_01/x.log", "2026/a/POS_02/y.log", "2026/b/POS_01/z.log",
                "2026/b/POS_01/notes.txt", "top.log"]:
        file = tmp / rel
        file.parent.mkdir(parents=True, exist_ok=True)
        file.write_text("test")
    found = find_log_files(tmp)

assert sorted(p.name for p in found) == ["top.log", "x.log", "y.log", "z.log"]
assert all(isinstance(p, Path) for p in found)
print("find_log_files OK ->", sorted(p.name for p in found))

find_log_files OK -> ['top.log', 'x.log', 'y.log', 'z.log']


### 2.2 Recursive traversal of the nested config
The JSON can be any mix of dicts and lists, so the function branches on the *type* of the current node:

* **dict** – if it has `"active": true` and a string `"id"`, record the id; then recurse into every value.
* **list** – recurse into every element.
* **anything else** (str, int, bool, None) – base case, nothing to collect.

The check `is True` makes sure only a real boolean `true` counts (not e.g. the string `"yes"`).

In [19]:
def get_active_registers(node) -> set[str]:
    """Return the ids of all registers with "active": true anywhere in a nested dict/list structure."""
    active: set[str] = set()
    if isinstance(node, dict):
        if node.get("active") is True and isinstance(node.get("id"), str):
            active.add(node["id"])
        for value in node.values():
            active |= get_active_registers(value)
    elif isinstance(node, list):
        for element in node:
            active |= get_active_registers(element)
    return active    # scalars: base case -> empty set

In [20]:
example_config = {
    "company": "Campus Coffee",
    "regions": [{
        "region_name": "Main Campus",
        "stores": [{
            "name": "Library Cafe",
            "registers": [
                {"id": "POS_01", "active": True},
                {"id": "POS_02", "active": False}],
        }],
    }],
}
assert get_active_registers(example_config) == {"POS_01"}

with open(DATA_DIR / "store_config.json", encoding="utf-8") as f:
    store_config = json.load(f)
print("get_active_registers OK -> active in store_config.json:", sorted(get_active_registers(store_config)))

get_active_registers OK -> active in store_config.json: ['POS_01', 'POS_02', 'POS_03']


## Task 3 – Cleaning and analysis with NumPy

### 3.1 Removing price outliers
1. Convert to a 1-D float array (`np.asarray(..., dtype=float).ravel()`).
2. Boolean mask `|x − μ| > 3σ` (population standard deviation).
3. Replace the flagged values by the **median of the valid prices** – the median is robust and not affected by the glitches that are being removed.

Everything is vectorised; there are no Python loops over the prices. The input array is not modified (we work on a copy).

In [21]:
def remove_price_outliers(prices, n_std: float = OUTLIER_N_STD) -> np.ndarray:
    """Replace values with |x - mean| > n_std * std by the median of the remaining (valid) values."""
    values = np.asarray(prices, dtype=float).ravel()
    if values.size == 0:
        return values.copy()

    mean, std = values.mean(), values.std()          # ddof=0 (population std)
    is_outlier = np.abs(values - mean) > n_std * std

    cleaned = values.copy()
    if is_outlier.any():
        cleaned[is_outlier] = np.median(values[~is_outlier])
    return cleaned

In [22]:
rng = np.random.default_rng(0)
prices = np.append(rng.normal(5, 1, 200), [999.0, 999.0, 999.0])

cleaned = remove_price_outliers(prices)
assert cleaned.shape == prices.shape and cleaned.ndim == 1
assert not (cleaned == 999.0).any()
assert np.allclose(cleaned[-3:], np.median(prices[prices < 999]))
assert np.array_equal(cleaned[:-3], prices[:-3])      # valid values untouched
assert prices[-1] == 999.0                           # input not modified
assert remove_price_outliers(list(prices)).shape == prices.shape   # also accepts lists
print("remove_price_outliers OK")

# Known limitation of the 3-sigma rule: with n <= 10 values it can never flag anything,
# because the largest possible z-score is (n-1)/sqrt(n) < 3.
print("n=4 :", remove_price_outliers([3.0, 4.0, 5.0, 999.0]))

remove_price_outliers OK
n=4 : [  3.   4.   5. 999.]


### 3.2 Refund ratio
Element-wise `refund_count / (sales_count + ε)` with `ε = 10⁻⁶`. Only NumPy array operations are used; the ε in the denominator prevents division by zero when an item has no sales.

In [23]:
def compute_refund_ratio(sales_count: np.ndarray, refund_count: np.ndarray) -> np.ndarray:
    """Element-wise refund ratio: refund_count / (sales_count + EPSILON)."""
    sales = np.asarray(sales_count, dtype=float)
    refunds = np.asarray(refund_count, dtype=float)
    return refunds / (sales + EPSILON)

In [24]:
ratio = compute_refund_ratio(np.array([10, 0, 5, 4]), np.array([1, 0, 5, 0]))
assert np.allclose(ratio, [0.1, 0.0, 1.0, 0.0])
assert np.isfinite(ratio).all()                      # no division-by-zero problems
print("compute_refund_ratio OK ->", ratio.round(4))

compute_refund_ratio OK -> [0.1 0.  1.  0. ]


## Task 4 – Data analysis with Pandas

| Function | What it does |
|---|---|
| `build_dataframe` | records → DataFrame, `timestamp` → `datetime64` index, sorted chronologically |
| `resample_hourly_revenue` | `groupby("item")` + `resample("1h")` + `sum()` of price → Series with a `(item, hour)` MultiIndex |
| `create_status_pivot` | pivot table: rows = item, columns = status (`SUCCESS`, `REFUND`, `CANCELLED`), values = summed price |
| `item_refund_ratios` | counts `SUCCESS`/`REFUND` per item and calls `compute_refund_ratio` from Task 3 |

Passing `columns=` to the DataFrame constructor makes the functions work even for an empty record list.

In [ ]:
TRANSACTION_COLUMNS = ["timestamp", "register_id", "item", "price", "currency", "status"]
STATUS_ORDER = ["SUCCESS", "REFUND", "CANCELLED"]


def build_dataframe(records: list[dict]) -> pd.DataFrame:
    """List of transaction dicts -> DataFrame indexed by (sorted) datetime timestamp."""
    df = pd.DataFrame(records, columns=TRANSACTION_COLUMNS)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    return df.set_index("timestamp").sort_index()


def resample_hourly_revenue(df: pd.DataFrame) -> pd.Series:
    """Total price per item in 1-hour windows."""
    return df.groupby("item")["price"].resample("1h").sum().rename("revenue")


def create_status_pivot(df: pd.DataFrame) -> pd.DataFrame:
    """Total revenue per item (rows) across the status categories (columns)."""
    pivot = df.pivot_table(values="price", index="item", columns="status",
                           aggfunc="sum", fill_value=0.0)
    return pivot.reindex(columns=STATUS_ORDER, fill_value=0.0)    # always all three columns


def item_refund_ratios(df: pd.DataFrame) -> pd.DataFrame:
    """Refund ratio per item = refunds / (successful sales + epsilon)."""
    counts = df.groupby(["item", "status"]).size().unstack(fill_value=0)
    counts = counts.reindex(columns=["SUCCESS", "REFUND"], fill_value=0)

    ratio = compute_refund_ratio(counts["SUCCESS"].to_numpy(), counts["REFUND"].to_numpy())
    return pd.DataFrame(
        {"sales_count": counts["SUCCESS"], "refund_count": counts["REFUND"], "refund_ratio": ratio},
        index=counts.index,
    )

In [ ]:
demo_records = [
    {"timestamp": "2026-10-15 08:30:00", "register_id": "POS_01", "item": "Latte",  "price": 4.5, "currency": "EUR", "status": "SUCCESS"},
    {"timestamp": "2026-10-15 07:10:00", "register_id": "POS_01", "item": "Latte",  "price": 4.0, "currency": "EUR", "status": "SUCCESS"},
    {"timestamp": "2026-10-15 09:05:00", "register_id": "POS_02", "item": "Latte",  "price": 4.5, "currency": "NOK", "status": "REFUND"},
    {"timestamp": "2026-10-15 07:40:00", "register_id": "POS_02", "item": "Muffin", "price": 3.5, "currency": "USD", "status": "SUCCESS"},
]
demo_df = build_dataframe(demo_records)
assert isinstance(demo_df.index, pd.DatetimeIndex) and demo_df.index.is_monotonic_increasing

demo_hourly = resample_hourly_revenue(demo_df)
assert demo_hourly[("Latte", pd.Timestamp("2026-10-15 08:00"))] == 4.5
assert demo_hourly[("Muffin", pd.Timestamp("2026-10-15 07:00"))] == 3.5

demo_pivot = create_status_pivot(demo_df)
assert list(demo_pivot.columns) == STATUS_ORDER
assert demo_pivot.loc["Latte", "SUCCESS"] == 8.5 and demo_pivot.loc["Latte", "REFUND"] == 4.5
assert demo_pivot.loc["Muffin", "REFUND"] == 0.0

demo_ratios = item_refund_ratios(demo_df)
assert np.isclose(demo_ratios.loc["Latte", "refund_ratio"], 0.5)      # 1 / (2 + eps)
assert demo_ratios.loc["Muffin", "refund_ratio"] == 0.0
print("Task 4 functions OK")
demo_pivot